# YOLO ポーズ推定サンプル

## YOLO ポーズ推定

このノートブックは **Ultralytics YOLO** を使った人体ポーズ推定のサンプルです。

```
画像（URL またはファイル）→ YOLO ポーズ推定器 → アノテーション済み画像 + キーポイント + 信頼スコア
```

検出（バウンディングボックスのみ）とは異なり、ポーズ推定は検出した各人物の**身体キーポイント**（目・肩・肘・手首・腰・膝・足首など）を位置推定します。

### 使用可能なモデル

| モデル | パラメータ数 | 速度 |
|-------|------------|------|
| `yolo11x-pose.pt` | 58.8 M | 最も遅い / 最も高精度 |
| `yolo11l-pose.pt` | 26.2 M | — |
| `yolo11m-pose.pt` | 20.9 M | — |
| `yolo11s-pose.pt` |  9.9 M | — |
| `yolo11n-pose.pt` |  2.9 M | 最も速い / 最も軽量 |

writefile セル内の `MODEL_NAME` を変更することでモデルを切り替えられます。

📄 [Ultralytics YOLO ドキュメント](https://docs.ultralytics.com/)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/CV-Samples/yolo"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls

In [ ]:
# Install Ultralytics
!pip install ultralytics -q

import ultralytics
ultralytics.checks()  # check environment

In [ ]:
# Download sample images from GitHub
import os

# Image files to download
IMAGE_FILES = [
    'lograstudio-yoga-3053487_640.jpg',
]

BASE_URL = 'https://raw.githubusercontent.com/mastnk/cv-samples/main/yolo'
for fname in IMAGE_FILES:
    url = f'{BASE_URL}/{fname}'
    dest = f'{PROJECT_PATH}/{fname}'
    if not os.path.exists(dest):
        os.system(f'wget -q "{url}" -O "{dest}"')
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

%cd "{PROJECT_PATH}"
!ls


## 独自画像を使うには

画像を用意する方法は 2 つあります。

**① URL を指定する**  
スクリプト実行時に `--url` フラグに直接画像 URL を渡します：
```
%run pose.py --url https://cdn.pixabay.com/photo/2015/08/10/18/01/dancing-882940_640.jpg
```

**② 画像ファイルを `PROJECT_PATH/` に置く**  
画像ファイルを `PROJECT_PATH/` に配置してから `--file` または `--dir` で実行します。

アップロードは **Google Drive** 経由が簡単です：  
[drive.google.com](https://drive.google.com) を開き、`マイドライブ / CV-Samples / yolo/` に移動してファイルをドラッグ＆ドロップするだけ。  
マウント済みの Drive を通じて Colab から即座にアクセスできます。追加のアップロード手順は不要です。

```
%run pose.py --file your_image.jpg   # ファイル 1 枚
%run pose.py --dir  .                # フォルダ内の全画像
```

## モデルを選択するには

下の writefile セル内の `MODEL_NAME` を変更してモデルを切り替えます。  
`MODEL_NAME = ...` の行が複数ある場合、**有効になるのは最後の 1 行だけ**です。

```python
# MODEL_NAME = 'yolo11x-pose.pt'  # 58.8M params — most accurate, slowest
# MODEL_NAME = 'yolo11l-pose.pt'  # 26.2M params
# MODEL_NAME = 'yolo11m-pose.pt'  # 20.9M params
# MODEL_NAME = 'yolo11s-pose.pt'  #  9.9M params
MODEL_NAME   = 'yolo11n-pose.pt'  #  2.9M params — fastest, lightest  ← active
```

モデルが大きいほど精度は高くなりますが、実行時間も長くなります。まずは `yolo11n-pose.pt` で試してみましょう。

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile pose.py
"""YOLO Pose Estimation — command-line interface.

Usage:
  %run pose.py --url  <image_url>  [--disp] [--conf FLOAT]
  %run pose.py --file <image_path> [--disp] [--conf FLOAT]
  %run pose.py --dir  <image_dir>  [--disp] [--conf FLOAT]
"""

import argparse
import glob
import os

from ultralytics import YOLO
from PIL import Image
import requests
from io import BytesIO

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME   = 'yolo11n-pose.pt'  #  2.9M params — fastest / lightest
# MODEL_NAME = 'yolo11s-pose.pt'  #  9.9M params
# MODEL_NAME = 'yolo11m-pose.pt'  # 20.9M params
# MODEL_NAME = 'yolo11l-pose.pt'  # 26.2M params
# MODEL_NAME = 'yolo11x-pose.pt'  # 58.8M params — most accurate

PROJECT_PATH = '/content/drive/MyDrive/CV-Samples/yolo'

# ---- Model loading -----------------------------------------------
model = YOLO(MODEL_NAME)

# ---- Display helper ----------------------------------------------
def show(annotated, label: str, disp: bool) -> None:
    """When --disp is active, display the annotated image then print the filename/URL."""
    if disp:
        if _has_ipy:
            ipy_display(annotated)
        print(label)

# ---- Functions ---------------------------------------------------
def pose_url(url: str, conf: float = 0.25, disp: bool = False):
    """Download an image from a URL and estimate pose."""
    image     = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    results   = model.predict(image, conf=conf, verbose=False)
    annotated = Image.fromarray(results[0].plot()[:, :, ::-1])  # BGR -> RGB
    show(annotated, url, disp)
    boxes = results[0].boxes
    names = results[0].names
    for box in boxes:
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        print(f'  {names[cls_id]:<20} {conf_val*100:.1f}%')
    return results


def pose_file(path: str, conf: float = 0.25, disp: bool = False):
    """Estimate pose in a single local image file."""
    image     = Image.open(path).convert('RGB')
    results   = model.predict(image, conf=conf, verbose=False)
    annotated = Image.fromarray(results[0].plot()[:, :, ::-1])  # BGR -> RGB
    show(annotated, path, disp)
    boxes = results[0].boxes
    names = results[0].names
    for box in boxes:
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        print(f'  {names[cls_id]:<20} {conf_val*100:.1f}%')
    return results


def pose_dir(directory: str, conf: float = 0.25, disp: bool = False):
    """Estimate pose in all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    for path in filepaths:
        pose_file(path, conf, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='YOLO Pose Estimation')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',  type=str,              help='Image URL for pose estimation')
group.add_argument('--file', type=str,              help='Path to a single image file')
group.add_argument('--dir',  type=str,              help='Directory of images')
parser.add_argument('--conf', type=float, default=0.25, help='Confidence threshold (default: 0.25)')
parser.add_argument('--disp', action='store_true',      help='Display annotated image and filename/URL before results')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    pose_url(args.url,   conf=args.conf, disp=args.disp)
elif args.file:
    pose_file(args.file, conf=args.conf, disp=args.disp)
elif args.dir:
    pose_dir(args.dir,   conf=args.conf, disp=args.disp)


## `pose.py` の使い方

上の `%%writefile` セルを実行すると、`pose.py` が `PROJECT_PATH` に保存されます。  
`--disp` でのインライン画像表示を有効にするため、`!python` ではなく **`%run`** で実行してください。

```
%run pose.py --url  <画像URL>        # リモート画像のポーズ推定
%run pose.py --file <画像パス>       # ローカルファイル 1 枚のポーズ推定
%run pose.py --dir  <ディレクトリ>  # フォルダ内の全画像をポーズ推定
```

**オプション引数**

| フラグ | デフォルト | 説明 |
|--------|-----------|------|
| `--disp` | オフ | 結果の前にアノテーション済み画像とファイル名 / URL を表示 |
| `--conf <f>` | `0.25` | 検出の信頼スコア閾値（0.0 〜 1.0） |

**実行例**

```bash
# リモート画像のポーズ推定（表示あり）
%run pose.py --url https://cdn.pixabay.com/photo/2015/08/10/18/01/dancing-882940_640.jpg --disp

# ファイル 1 枚、信頼スコア閾値を下げてポーズ推定
%run pose.py --file lograstudio-yoga-3053487_640.jpg --conf 0.1 --disp

# フォルダ内の全画像をポーズ推定（各画像を表示）
%run pose.py --dir . --disp
```

**出力形式（`--disp` あり）**

```
<キーポイント付きアノテーション済み画像をインライン表示>
lograstudio-yoga-3053487_640.jpg
  person               91.2%
```

## 実行方法

`pose.py` は `!python` ではなく **`%run`** で実行してください。Colab カーネル内で実行されるため、`--disp` でのインライン画像表示が有効になります。

| モード | フラグ | 説明 |
|--------|--------|------|
| URL から | `--url <url>` | リモート画像を取得してポーズ推定 |
| ファイル 1 枚 | `--file <パス>` | ローカル画像 1 枚をポーズ推定 |
| ディレクトリ | `--dir <パス>` | フォルダ内の全画像をポーズ推定 |

`--disp` を追加すると、結果の前に各アノテーション済み画像（キーポイント付き）とファイル名 / URL が表示されます。  
`--conf <f>` で信頼スコア閾値を変更できます（デフォルト: 0.25）。

In [ ]:
# Execute: estimate pose in an image from URL (with display)
%cd "{PROJECT_PATH}"
%run pose.py --disp --url https://cdn.pixabay.com/photo/2015/08/10/18/01/dancing-882940_640.jpg


In [ ]:
# Execute: estimate pose in a single local image file (with display)
%cd "{PROJECT_PATH}"
%run pose.py --disp --file lograstudio-yoga-3053487_640.jpg


In [ ]:
# Execute: estimate pose in all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run pose.py --disp --dir .


💾 **ノートブックを保存するのを忘れずに！**

作業内容を保持するため、閉じる前に **ファイル → ドライブにコピーを保存** を実行してください。